# 3-Way Statistical Model Comparison
Evaluating MARL vs Sigmoid vs Ungated ViT on ImageNet-100 and CIFAR-10.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from datasets import load_dataset
from torch.utils.data import DataLoader
from statsmodels.stats.contingency_tables import mcnemar
from tqdm.auto import tqdm
import os
import time
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Import architectures
import marl_model
import sigmoid
import ungated_model


In [ ]:
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=0.):
        self.std = std
        self.mean = mean
    def __call__(self, tensor):
        if self.std == 0: return tensor
        return tensor + torch.randn(tensor.size()) * self.std + self.mean

class HFImageNetDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform=None):
        self.hf_split = hf_split
        self.transform = transform
    def __len__(self):
        return len(self.hf_split)
    def __getitem__(self, idx):
        item = self.hf_split[idx]
        image = item['image']
        if image.mode != 'RGB': image = image.convert('RGB')
        if self.transform: image = self.transform(image)
        return image, item['label']

class HFCifarDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform=None):
        self.hf_split = hf_split
        self.transform = transform
    def __len__(self):
        return len(self.hf_split)
    def __getitem__(self, idx):
        item = self.hf_split[idx]
        image = item['img']
        if image.mode != 'RGB': image = image.convert('RGB')
        if self.transform: image = self.transform(image)
        return image, item['label']


In [ ]:
def get_loaders(noise_std=0.0):
    t_in100 = transforms.Compose([
        transforms.Resize((144, 144)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        AddGaussianNoise(0., noise_std)
    ])
    t_c10 = transforms.Compose([
        transforms.Resize((144, 144)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        AddGaussianNoise(0., noise_std)
    ])
    
    print("Loading ImageNet-100...")
    hf_in100_val = load_dataset('clane9/imagenet-100', split='validation')
    in100_loader = DataLoader(HFImageNetDataset(hf_in100_val, t_in100), batch_size=128, shuffle=False, num_workers=0)
    
    print("Loading CIFAR-10...")
    hf_c10_val = load_dataset('cifar10', split='test')
    cifar_loader = DataLoader(HFCifarDataset(hf_c10_val, t_c10), batch_size=128, shuffle=False, num_workers=0)
    
    return in100_loader, cifar_loader

in100_loader, cifar_loader = get_loaders(noise_std=0.0)


In [ ]:
def measure_model_stats(name, model, loader, device):
    params = sum(p.numel() for p in model.parameters())
    model.eval()
    try:
        imgs, _ = next(iter(loader))
        imgs = imgs[:8].to(device)
        with torch.no_grad():
            for _ in range(5): model(imgs, deterministic=True)
        if 'cuda' in str(device): torch.cuda.synchronize()
        
        start = time.time()
        b_limit = 5
        count = 0
        with torch.no_grad():
            for i, (images, _) in enumerate(loader):
                model(images.to(device), deterministic=True)
                count += images.size(0)
                if i >= b_limit: break
        if 'cuda' in str(device): torch.cuda.synchronize()
        dur = time.time() - start
        fps = count / dur
        print(f'[{name}] Params: {params:,} | FPS: {fps:.2f}')
    except Exception as e:
        print(f"Error measuring stats for {name}: {e}")


In [ ]:
def run_pairwise_comparison(model_a, model_b, name_a, name_b, loader, dataset_name, device):
    print(f'\n--- {dataset_name}: {name_a} vs {name_b} ---')
    a_ok_b_ok, a_ok_b_no, a_no_b_ok, a_no_no = 0, 0, 0, 0
    total = 0
    correct_a, correct_b = 0, 0
    
    model_a.eval(); model_b.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f'{name_a} vs {name_b}'):
            imgs, labels = imgs.to(device), labels.to(device)
            out_a = model_a(imgs, deterministic=True)['logits'].argmax(-1)
            out_b = model_b(imgs, deterministic=True)['logits'].argmax(-1)
            
            mask_a = (out_a == labels)
            mask_b = (out_b == labels)
            
            correct_a += mask_a.sum().item()
            correct_b += mask_b.sum().item()
            total += labels.size(0)
            
            a_ok_b_ok += (mask_a & mask_b).sum().item()
            a_ok_b_no += (mask_a & ~mask_b).sum().item()
            a_no_b_ok += (~mask_a & mask_b).sum().item()
            a_no_no   += (~mask_a & ~mask_b).sum().item()
            
    acc_a = 100 * correct_a / total
    acc_b = 100 * correct_b / total
    print(f'{name_a} Acc: {acc_a:.2f}% | {name_b} Acc: {acc_b:.2f}%')
    
    table = [[a_ok_b_ok, a_ok_b_no], [a_no_b_ok, a_no_no]]
    res = mcnemar(table, exact=False, correction=True)
    print(f'McNemar Stats: chi2={res.statistic:.2f}, p={res.pvalue:.4e}')
    
    plt.figure(figsize=(4,3))
    sns.heatmap(table, annot=True, fmt='d', cmap='Greens', 
                xticklabels=[f'{name_b} OK', f'{name_b} ERR'], 
                yticklabels=[f'{name_a} OK', f'{name_a} ERR'])
    plt.title(f'{dataset_name}: {name_a} vs {name_b}')
    plt.show()
    return res.pvalue
run_mcnemar_test = run_pairwise_comparison

In [ ]:
def load_models(dataset_type='imagenet', device='cuda'):
    dim = 256; depth = 16; heads = 8; patch = 12
    n_cls = 100 if dataset_type == 'imagenet' else 10
    
    m_marl = marl_model.ViT(image_size=144, patch_size=patch, num_classes=n_cls, d_model=dim, depth=depth, head=heads).to(device)
    m_sig  = sigmoid.ViT(image_size=144, patch_size=patch, num_classes=n_cls, d_model=dim, depth=depth, head=heads).to(device)
    m_ung  = ungated_model.ViT(image_size=144, patch_size=patch, num_classes=n_cls, d_model=dim, depth=depth, head=heads).to(device)
    
    if dataset_type == 'imagenet':
        m_marl.load_state_dict(torch.load('checkpoints/0.1,1.0/vit_v2_stage_3_epoch7.pth', map_location=device))
        m_sig.load_state_dict(torch.load('checkpoints_sigmoid/sigmoid_vit_epoch13.pth', map_location=device))
        m_ung.load_state_dict(torch.load('ungated/vit_epoch15.pth', map_location=device))
    else:
        m_marl.load_state_dict(torch.load('vit_cifar_v2_stage_3_final.pth', map_location=device))
        m_sig.load_state_dict(torch.load('checkpoints_sigmoid/cifar_vit_epoch19.pth', map_location=device))
        m_ung.load_state_dict(torch.load('ungated/vit_cifar_epoch20.pth', map_location=device))
        
    return m_marl, m_sig, m_ung


In [ ]:
print("=== STARTING 3-WAY COMPARISON ===")
for dset in ['imagenet', 'cifar']:
    name = "ImageNet-100" if dset=='imagenet' else "CIFAR-10"
    loader = in100_loader if dset=='imagenet' else cifar_loader
    
    m_marl, m_sig, m_ung = load_models(dset, device)
    
    measure_model_stats('MARL', m_marl, loader, device)
    measure_model_stats('Sigmoid', m_sig, loader, device)
    measure_model_stats('Ungated', m_ung, loader, device)
    
    run_pairwise_comparison(m_marl, m_sig, 'MARL', 'Sigmoid', loader, name, device)
    run_pairwise_comparison(m_marl, m_ung, 'MARL', 'Ungated', loader, name, device)
    run_pairwise_comparison(m_sig, m_ung, 'Sigmoid', 'Ungated', loader, name, device)
